## Library + API keys

In [ ]:
!pip freeze

In [1]:
!pip install -qU langchain-unstructured unstructured-client unstructured langchain_openai ragas faiss-cpu
!pip install langchain-community<0.4.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 48.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.6/160.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 31.5 MB/s eta 0:00:0

In [2]:
# urls for kubernetes documentation
urls = [
    "https://kubernetes.io/docs/concepts/overview/",
    "https://kubernetes.io/docs/concepts/architecture/",
    "https://kubernetes.io/docs/concepts/containers/",
    "https://kubernetes.io/docs/concepts/workloads/",
    "https://kubernetes.io/docs/concepts/storage/"
]

In [3]:
# download NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [4]:
from google.colab import userdata
openai_api = userdata.get('genai_course')

import os
os.environ['OPENAI_API_KEY'] = openai_api

# Data + RAG

In [5]:
# Import required functions / classes
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_5204/908222110.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [6]:
# Load the documents and split them
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
)
loader = UnstructuredURLLoader(urls=urls)
documents = loader.load_and_split(text_splitter)

In [7]:
# Embed the chunks
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents, embeddings)

In [8]:
def rag(query, k = 5):
  # Retrieving the context
  retrieved_docs = db.similarity_search(query, k = k)

  # Combined all the context from the the retrieved docs
  combined_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

  # Define the prompt
  prompt = f""" based on this context: {combined_text} answer the query: {query}"""

  # Call the LLM model
  model = ChatOpenAI(model = "gpt-5-nano",
                    temperature = 0)
  response_text = model.invoke(prompt)

  # Return output
  return response_text.content, [txt.page_content for txt in retrieved_docs]

# Text the function
query = "What is the overview of kubernetes?"
rag(query)

('Kubernetes is an open-source platform for managing containerized workloads and services. It provides a framework to run distributed systems in a scalable, automated, and resilient way, using declarative configuration and a rich ecosystem of tools.\n\nKey points:\n- What it does: service discovery and load balancing; storage orchestration; automated rollouts and rollbacks; automatic bin packing; self-healing; secret and configuration management; horizontal scaling; and more.\n- Architecture (high level): a cluster consists of a control plane and worker nodes. The control plane (kube-apiserver, etcd, kube-scheduler, kube-controller-manager) makes global decisions and responds to events, while the nodes run Pods that host your applications.\n- Not a PaaS: Kubernetes is not a monolithic platform that provides all application services or CI/CD; it provides building blocks to run and manage containers, and you plug in logging, monitoring, databases, etc.\n- Extensibility: supports CustomRe

# Generate Synthetic data

In [9]:
from ragas.llms import llm_factory
from ragas.testset import TestsetGenerator
from openai import OpenAI
from ragas.embeddings.base import embedding_factory

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
client = OpenAI()

In [ ]:
# LLM and Embeddings
generator_llm = llm_factory(model = "gpt-5.5",
                            client = client)
generator_embeddings = embedding_factory('openai', model='text-embedding-3-large', client=client)

In [ ]:
# (optional) remove empty chunks
chunks = [d for d in documents if d.page_content and d.page_content.strip()]

# Test Set Generator
generator = TestsetGenerator(llm = generator_llm,
                             embedding_model = generator_embeddings)

dataset = generator.generate_with_chunks(
    chunks=chunks,
    testset_size=10, # Reduced from 30 to 10 to avoid hitting token limits
    raise_exceptions=False,  # don't crash the whole run if a single task fails
)

dataset.to_pandas().head()

In [ ]:
# Put the test data as a pandas df
test_df = dataset.to_pandas()
test_df.head()

In [ ]:
# Answer each user_input with our RAG and put it in the df
answers_gen = []
context_gen = []
for user_input in test_df['user_input']:
  answer, context = rag(user_input)
  answers_gen.append(answer)
  context_gen.append(context)

# Store it in the pandas df
test_df['answer'] = answers_gen
test_df['context'] = context_gen

In [ ]:
# Preview
test_df.head()

# Rouge Score

In [ ]:
!pip install -q rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
# Import classes and libraries
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RougeScore
import numpy as np

/tmp/ipykernel_552/1503278409.py:3: DeprecationWarning: Importing RougeScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import RougeScore
  from ragas.metrics import RougeScore


https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/traditional/#rouge-score

In [ ]:
# Rouge Score
# Getting the rouge score for each test set row
rouge_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer']
  )

  # Compute the rouge score
  scorer = RougeScore()
  rouge_score = await scorer.single_turn_ascore(sample)

  rouge_score_list.append(rouge_score)

In [ ]:
# Print the output
print(rouge_score_list)
print(f"The mean rouge score is {np.mean(rouge_score_list)}")

[0.4414414414414415, 0.04651162790697675, 0.13714285714285715, 0.09561752988047809, 0.2385685884691849, 0.09888357256778312, 0.17133956386292837, 0.09736632083000797, 0.19471947194719474, 0.09417596034696406, 0.1399548532731377, 0.2097560975609756]
The mean rouge score is 0.16378982376916082


# LLM based evaluation

In [ ]:
# Define evaluation llm and embedding model
eval_llm = llm_factory(model = "gpt-4o",
                       max_tokens = 16000,
                       client = client)
eval_embeddings = embedding_factory('openai', model='text-embedding-3-large', client=client)

## Simple Scoring

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/general_purpose/#simple-criteria-scoring

In [ ]:
from ragas.metrics import SimpleCriteriaScore

/tmp/ipykernel_552/3723547320.py:1: DeprecationWarning: Importing SimpleCriteriaScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SimpleCriteriaScore
  from ragas.metrics import SimpleCriteriaScore


In [ ]:
# Define the simple scorer
simple_scorer = SimpleCriteriaScore(
    name = "simple scorer",
    definition = "Score 0 to 5 by similarity",
    llm = eval_llm
)

In [ ]:
# Get the simple score for every row
simple_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'])
  simple_score = await simple_scorer.single_turn_ascore(sample)
  simple_score_list.append(simple_score)

# Print the outputs
print(simple_score_list)
print(f"The mean simple score is {np.mean(simple_score_list)}")

[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
The mean simple score is 5.0


In [ ]:
# inspect the data
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'])
  print(sample)

user_input='Whatt rolee did Gooogle play in the developement of Kubernetess, and howw does it contribute to the management of containerized workloads?' retrieved_contexts=None reference_contexts=None retrieved_context_ids=None reference_context_ids=None response="- Google's role in Kubernetes development\n  - Google founded and open-sourced Kubernetes in 2014.\n  - The project builds on more than 15 years of Google’s experience running production workloads at scale.\n  - This background informs Kubernetes’ design and capabilities, while the broader community contributes ideas and tooling.\n\n- How Kubernetes contributes to managing containerized workloads\n  - Provides a framework to run distributed systems resiliently, handling scaling and failover.\n  - Supports deployment patterns, automated rollouts and rollbacks, and canary deployments.\n  - Offers service discovery and load balancing, storage orchestration, and automatic bin packing to fit containers onto nodes.\n  - Includes sel

## Rubrics Scoring

In [ ]:
rubrics = {
    "score1_description": "The response is entirely incorrect and fails to address any aspect of the reference.",
    "score2_description": "The response contains partial accuracy but includes major errors or significant omissions that affect its relevance to the reference.",
    "score3_description": "The response is mostly accurate but lacks clarity, thoroughness, or minor details needed to fully address the reference.",
    "score4_description": "The response is accurate and clear, with only minor omissions or slight inaccuracies in addressing the reference.",
    "score5_description": "The response is completely accurate, clear, and thoroughly addresses the reference without any errors or omissions.",
}

In [ ]:
from ragas.metrics import RubricsScore

/tmp/ipykernel_552/4048792130.py:1: DeprecationWarning: Importing RubricsScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import RubricsScore
  from ragas.metrics import RubricsScore


In [ ]:
# Define the rubrics scorer
rubrics_scorer = RubricsScore(
    rubrics = rubrics,
    llm = eval_llm
)

In [ ]:
# Compute the rubrics score for each row
rubrics_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'])
  rubrics_score = await rubrics_scorer.single_turn_ascore(sample)
  rubrics_score_list.append(rubrics_score)

# Print the outputs
print(rubrics_score_list)
print(f"The mean rubrics score is {np.mean(rubrics_score_list)}")


[4, 5, 5, 5, 4, 5, 4, 4, 5, 4, 5, 4]
The mean rubrics score is 4.5


# RAG-Specific Metrics

## Factual Correctness

In [ ]:
from ragas.metrics._factual_correctness import FactualCorrectness

In [ ]:
# Define the scorer
factual_scorer = FactualCorrectness(
    llm = eval_llm
)

In [ ]:
# Compute the factual correctness for each row
factual_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'])
  factual_score = await factual_scorer.single_turn_ascore(sample)
  factual_score_list.append(factual_score)

# Print the outputs
print(factual_score_list)
print(f"The mean factual score is {np.mean(factual_score_list)}")


[np.float64(0.73), np.float64(0.08), np.float64(0.49), np.float64(0.18), np.float64(0.5), np.float64(0.4), np.float64(0.65), np.float64(0.06), np.float64(0.46), np.float64(0.14), np.float64(0.24), np.float64(0.48)]
The mean factual score is 0.3675


## Semantic Similarity

In [ ]:
from ragas.metrics import SemanticSimilarity

In [ ]:
# Get the semantics evaluator
semantic_scorer = SemanticSimilarity(
    embeddings = eval_embeddings
)

In [ ]:
# Compute the similatiry for each row
semantic_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'])
  semantic_score = await semantic_scorer.single_turn_ascore(sample)
  semantic_score_list.append(semantic_score)

# Print the outputs
print(semantic_score_list)
print(f"The mean semantic score is {np.mean(semantic_score_list)}")

[0.9394204793107228, 0.9056888390364272, 0.9653140366986089, 0.9457320010758579, 0.964971554893908, 0.966151567948898, 0.9609037207282072, 0.9949685385236932, 0.9484119939749569, 0.9110613958343153, 0.9502632868908938, 0.9657694820858482, 0.9612548109211636, 0.9598681446105665, 0.9305181809041412]
The mean semantic score is 0.9513532022292139


## Context Precision

In [ ]:
from ragas.metrics import LLMContextPrecisionWithReference

In [ ]:
# Get the context scorer
context_scorer = LLMContextPrecisionWithReference(
    llm = eval_llm
)

In [ ]:
test_df.head(1)

,user_input,reference_contexts,reference,synthesizer_name,answer,context
0,What is the role of the Route controller in Ku...,[There are many different types of controllers...,The Route controller is responsible for settin...,single_hop_specific_query_synthesizer,The Route controller in the Kubernetes **cloud...,[There are many different types of controllers...


In [ ]:
# # Compute the Precision for each row
# context_score_list = []
# for row in test_df.to_dict(orient = "records"):
#   sample = SingleTurnSample(
#       user_input = row['user_input'],
#       reference = row['reference'],
#       response = row['answer'],
#       retrieved_contexts = row['context'])
#   context_score = await context_scorer.single_turn_ascore(sample)
#   context_score_list.append(context)

# # Print the results
# print(context_score_list)
# print(f"The mean context score is {np.mean(context_score_list)}")

In [ ]:
# Compute the Precision for each row
context_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'],
      retrieved_contexts = row['context'])
  context_score = await context_scorer.single_turn_ascore(sample)
  context_score_list.append(context_score)

# Print the results
print(context_score_list)
print(f"The mean context score is {np.mean(context_score_list)}")

[0.9999999999, 0.9999999999, 0.49999999995, 0.9999999999, 0.9999999999, 0.9999999999, 0.9999999999, 0.9999999999, 0.9999999999, 0.7499999999625, 0.9999999999, 0.49999999995, 0.9999999999, 0.999999999975, 0.9999999999]
The mean context score is 0.9166666665824998


## Context Recall

In [ ]:
from ragas.metrics import LLMContextRecall

In [ ]:
# Define the evaluator for recall
context_recall_scorer = LLMContextRecall(
    llm = eval_llm
)

In [ ]:
# Compute the recall score for each row
context_recall_score_list = []
for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'],
      retrieved_contexts = row['context'])
  context_recall_score = await context_recall_scorer.single_turn_ascore(sample)
  context_recall_score_list.append(context_recall_score)

# Print the results
print(context_recall_score_list)
print(f"The mean context recall score is {np.mean(context_recall_score_list)}")


[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
The mean context recall score is 1.0


## Response Relevancy

In [ ]:
from ragas.metrics import ResponseRelevancy

In [ ]:
# Define the response relevancy scorer
response_relevancy_scorer = ResponseRelevancy(
    llm = eval_llm,
    embeddings = eval_embeddings
)

In [ ]:
# Compute the Response relevancy
response_relevancy_score_list = []

for row in test_df.to_dict(orient = "records"):
  sample = SingleTurnSample(
      user_input = row['user_input'],
      reference = row['reference'],
      response = row['answer'],
      retrieved_contexts = row['context'])
  response_relevancy_score = await response_relevancy_scorer.single_turn_ascore(sample)
  response_relevancy_score_list.append(response_relevancy_score)

# Print the results
print(response_relevancy_score_list)
print(f"The mean response relevancy score is {np.mean(response_relevancy_score_list)}")
#

[np.float64(0.9822088911660561), np.float64(0.882382150014914), np.float64(1.0), np.float64(0.9440347023841191), np.float64(1.0), np.float64(0.8854612788718085), np.float64(1.0), np.float64(0.9584437988446192), np.float64(1.0000000000000009), np.float64(0.8055827162092463), np.float64(0.9499820115793248), np.float64(0.9844650933260448), np.float64(0.999998784180466), np.float64(0.9774632993583707), np.float64(0.9771101250295842)]
The mean response relevancy score is 0.9564755233976371
